In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 04:19:37 WARN Utils: Your hostname, codespaces-ccc6f6, resolves to a loopback address: 127.0.0.1; using 10.0.2.244 instead (on interface eth0)
26/03/02 04:19:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 04:19:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 04:19:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
import subprocess, os
print(subprocess.check_output(["java","-version"], stderr=subprocess.STDOUT).decode())
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

openjdk version "25.0.1" 2025-10-21 LTS
OpenJDK Runtime Environment Microsoft-12574222 (build 25.0.1+8-LTS)
OpenJDK 64-Bit Server VM Microsoft-12574222 (build 25.0.1+8-LTS, mixed mode, sharing)

JAVA_HOME: /usr/local/sdkman/candidates/java/current


In [6]:
import subprocess

# Show the offending repo file(s), then disable them
subprocess.run("ls -la /etc/apt/sources.list.d | sed -n '1,200p'", shell=True, check=False)
subprocess.run("grep -R \"dl.yarnpkg.com\" -n /etc/apt/sources.list /etc/apt/sources.list.d || true", shell=True, check=False)

# Disable yarn repo list (common filenames)
subprocess.run("sudo sh -c 'for f in /etc/apt/sources.list.d/*yarn*; do [ -e \"$f\" ] && mv \"$f\" \"$f.disabled\"; done'", shell=True, check=False)

# Also disable any list that still references yarnpkg
subprocess.run("sudo sh -c 'for f in /etc/apt/sources.list.d/*.list; do grep -q \"dl.yarnpkg.com\" \"$f\" && mv \"$f\" \"$f.disabled\"; done'", shell=True, check=False)

print("Disabled yarn repo list entries (if present).")

total 28
drwxr-xr-x 1 root root 4096 Nov 27 10:44 .
drwxr-xr-x 1 root root 4096 Oct 13 14:03 ..
-rw-r--r-- 1 root root  135 Nov 27 10:36 conda.list
-rw-r--r-- 1 root root  153 Nov 27 10:44 microsoft.list
-rw-r--r-- 1 root root 2552 Oct 13 14:09 ubuntu.sources
-rw-rw-r-- 1 root root  115 Nov 27 10:36 yarn.list
/etc/apt/sources.list.d/yarn.list:1:deb [arch=amd64 signed-by=/usr/share/keyrings/yarn-archive-keyring.gpg] https://dl.yarnpkg.com/debian/ stable main
Disabled yarn repo list entries (if present).


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("sanity").getOrCreate()
print("Spark:", spark.version)
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 04:23:15 WARN Utils: Your hostname, codespaces-ccc6f6, resolves to a loopback address: 127.0.0.1; using 10.0.2.244 instead (on interface eth0)
26/03/02 04:23:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 04:23:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 04:23:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 4.1.1


In [9]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-02 04:13:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.65, 52.85.39.117, 52.85.39.97, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M   112MB/s    in 0.6s    

2026-03-02 04:13:45 (112 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]

--2026-03-02 04:13:46--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.117, 52.85.39.97, 52.85.39.153, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.117|:443... connected.
HTTP request sent, awaiting respons

In [3]:
from pathlib import Path
import shutil

# Input parquet (your exact path)
yellow_path = "file://" + str(Path("/workspaces/dtc_module5_batching/yellow_tripdata_2025-11.parquet"))
df = spark.read.parquet(yellow_path)

# Output directory
out_dir = Path("/workspaces/dtc_module5_batching/yellow_2025_11_repart4")
if out_dir.exists():
    shutil.rmtree(out_dir)

# Repartition to 4 and write
(df.repartition(4)
   .write
   .mode("overwrite")
   .parquet("file://" + str(out_dir))
)

# Compute average size of part files (MB)
parquet_files = list(out_dir.rglob("*.parquet"))
sizes_mb = [p.stat().st_size / (1024 * 1024) for p in parquet_files]
avg_mb = sum(sizes_mb) / len(sizes_mb)

print("Q2 number of .parquet files:", len(parquet_files))
print("Q2 avg size (MB):", round(avg_mb, 2))
print("Q2 sizes (MB):", [round(x, 2) for x in sizes_mb])

openjdk version "25.0.1" 2025-10-21 LTS
OpenJDK Runtime Environment Microsoft-12574222 (build 25.0.1+8-LTS)
OpenJDK 64-Bit Server VM Microsoft-12574222 (build 25.0.1+8-LTS, mixed mode, sharing)

JAVA_HOME = /usr/local/sdkman/candidates/java/current


In [11]:
import os, urllib.request

yellow_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
zones_url  = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

yellow_file = "yellow_tripdata_2025-11.parquet"
zones_file  = "taxi_zone_lookup.csv"

if not os.path.exists(yellow_file):
    urllib.request.urlretrieve(yellow_url, yellow_file)

if not os.path.exists(zones_file):
    urllib.request.urlretrieve(zones_url, zones_file)

(yellow_file, zones_file)

('yellow_tripdata_2025-11.parquet', 'taxi_zone_lookup.csv')

In [12]:
from pyspark.sql import functions as F

# Detect correct pickup column name
pickup_col = "tpep_pickup_datetime" if "tpep_pickup_datetime" in df.columns else "pickup_datetime"

q3_count = (
    df.filter(F.to_date(F.col(pickup_col)) == F.lit("2025-11-15"))
      .count()
)

q3_count

NameError: name 'df' is not defined

In [13]:
from pyspark.sql import functions as F

pickup_col  = "tpep_pickup_datetime" if "tpep_pickup_datetime" in df.columns else "pickup_datetime"
dropoff_col = "tpep_dropoff_datetime" if "tpep_dropoff_datetime" in df.columns else "dropoff_datetime"

q4_max_hours = (
    df.select(
        ((F.unix_timestamp(F.col(dropoff_col)) - F.unix_timestamp(F.col(pickup_col))) / 3600.0).alias("trip_hours")
    )
    .agg(F.max("trip_hours").alias("max_hours"))
    .collect()[0]["max_hours"]
)

float(q4_max_hours)

NameError: name 'df' is not defined

In [14]:
zones = (
    spark.read.option("header", True)
    .csv("taxi_zone_lookup.csv")
    .select(
        F.col("LocationID").cast("int").alias("LocationID"),
        F.col("Zone")
    )
)

least_zone = (
    df.select(F.col("PULocationID").cast("int").alias("LocationID"))
      .join(zones, on="LocationID", how="left")
      .groupBy("Zone")
      .count()
      .orderBy(F.col("count").asc(), F.col("Zone").asc())
      .limit(1)
      .collect()[0]
)

least_zone["Zone"], least_zone["count"]

26/03/02 04:13:58 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: taxi_zone_lookup.csv.
java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3888)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3878)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3666)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:289)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:541)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.Data

UnsupportedOperationException: getSubject is not supported